# 🏏 IPL Match Predictor — Championship Edition
## Built for the IPL Hackathon | Goal: Lowest Log-Loss on Leaderboard

Welcome! This notebook uses **5 real datasets** to predict IPL match outcomes in 4 categories:
- `A_big` — Team A wins by a large margin
- `A_small` — Team A wins closely
- `B_big` — Team B wins comfortably
- `B_small` — Team B edges it

We go far beyond a basic model — we compute **rolling form, head-to-head records, powerplay strength, death-over firepower, and venue tendencies** to give the model the best possible signal before making a prediction.


In [ ]:
# Step 1: Install required libraries
# We need xgboost for gradient boosting and scikit-learn for calibration
!pip install xgboost scikit-learn pandas numpy -q
print("All dependencies ready!")

In [ ]:
# Step 2: Import everything we need
import pandas as pd
import numpy as np
import os, glob, warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, accuracy_score
from xgboost import XGBClassifier
from IPython.display import display

warnings.filterwarnings('ignore')
print("Imports done!")

## 📂 Dataset Overview

We are using **5 datasets** from the `DataSetting` Kaggle input folder.
Each one plays a specific role in our prediction pipeline:

| Dataset | Role |
|---|---|
| `match_summary.csv` | Core training data — past match outcomes with venue & team info |
| `train_IPL.csv` | Ball-by-ball data — 272K+ rows for powerplay & death-over features |
| `public_lb_matches.csv` | The 48 matches we must predict for the leaderboard |
| `schedule.csv` | Full fixture list — useful for upcoming match context |
| `sample_submission.csv` | The exact format our output must match |


In [ ]:
# Step 3: Load all 5 datasets
# The notebook auto-detects whether it's running on Kaggle or locally

def find_file(name):
    """Search for a file in both Kaggle input dirs and local data folder."""
    patterns = [
        f'/kaggle/input/**/{name}',
        f'./backend/data/{name}',
        f'./**/{name}'
    ]
    for p in patterns:
        matches = glob.glob(p, recursive=True)
        if matches:
            return matches[0]
    return None

# Load each dataset and confirm it was found
DATASETS = {
    'match_summary':  find_file('match_summary.csv'),
    'train_ipl':      find_file('train_IPL.csv'),
    'lb_matches':     find_file('public_lb_matches.csv'),
    'schedule':       find_file('schedule.csv'),
    'sample_sub':     find_file('sample_submission.csv'),
}

loaded = {}
for name, path in DATASETS.items():
    if path:
        loaded[name] = pd.read_csv(path, low_memory=False)
        print(f"[OK] {name}: {loaded[name].shape[0]} rows, {loaded[name].shape[1]} cols")
    else:
        print(f"[MISSING] {name} — check your Kaggle dataset is attached")


## 🔍 Understanding the Outcome Classes

Before we build the model, let's understand what we're predicting.
The 4 outcome classes are based on **who won** and **by how much**:
- A margin above a threshold = `_big`, below = `_small`

A balanced distribution across all 4 classes is ideal — let's verify.


In [ ]:
# Step 4: Explore the outcome distribution in our training data
ms = loaded['match_summary'].copy()
ms['date'] = pd.to_datetime(ms['date'])
ms = ms.sort_values('date').reset_index(drop=True)

print("Outcome distribution in match_summary:")
print(ms['outcome'].value_counts())
print()

print("Date range:", ms['date'].min().date(), "to", ms['date'].max().date())
print("Total matches:", len(ms))
print("Unique teams:", pd.concat([ms['team_a'], ms['team_b']]).nunique())
print("Unique venues:", ms['venue'].nunique())


## ⚙️ Feature Engineering — The Secret to Low Log-Loss

This is the most important step. We compute features that tell the model:
1. **Rolling win rate** — How well has each team been doing recently?
2. **Head-to-head record** — Does one team consistently dominate the other?
3. **Venue win rate** — Does Team A perform better at this specific ground?
4. **Powerplay strength** — From `train_IPL.csv`, how many runs does each team score in overs 1-6?
5. **Death-over firepower** — How explosive are they in overs 16-20?

> ⚠️ **No data leakage**: All rolling stats are calculated using only *past* matches.
> The model never sees future information — ensuring the leaderboard score matches training.


In [ ]:
# Step 5: Compute rolling historical features (no data leakage)

def compute_rolling_features(df):
    """
    Walk through matches chronologically.
    For each match, record historical stats BEFORE it happened.
    Then update the stats using the result of that match.
    This ensures zero leakage.
    """
    team_wins, team_total = {}, {}
    h2h = {}
    venue_wins, venue_total = {}, {}
    rows = []

    for _, row in df.iterrows():
        ta, tb, venue, outcome = row['team_a'], row['team_b'], row['venue'], row['outcome']

        # --- Win Rate (past matches only) ---
        wr_a = team_wins.get(ta, 0) / max(team_total.get(ta, 1), 1)
        wr_b = team_wins.get(tb, 0) / max(team_total.get(tb, 1), 1)

        # --- Head-to-Head ---
        key = tuple(sorted([ta, tb]))
        h2h_a = h2h.get(key, {}).get(ta, 0) / max(h2h.get(key, {}).get('total', 1), 1)

        # --- Venue Win Rate for Team A ---
        vk_a = (venue, ta)
        v_wr_a = venue_wins.get(vk_a, 0) / max(venue_total.get(venue, 1), 1)

        rows.append({
            'wr_a': wr_a, 'wr_b': wr_b,
            'wr_diff': wr_a - wr_b,
            'h2h_a': h2h_a, 'h2h_edge': h2h_a - 0.5,
            'v_wr_a': v_wr_a
        })

        # --- Update history with this match's result ---
        win_a = 'A' in outcome
        team_wins[ta] = team_wins.get(ta, 0) + (1 if win_a else 0)
        team_wins[tb] = team_wins.get(tb, 0) + (0 if win_a else 1)
        team_total[ta] = team_total.get(ta, 0) + 1
        team_total[tb] = team_total.get(tb, 0) + 1
        if key not in h2h:
            h2h[key] = {'total': 0, ta: 0, tb: 0}
        h2h[key]['total'] += 1
        winner = ta if win_a else tb
        h2h[key][winner] = h2h[key].get(winner, 0) + 1
        venue_wins[vk_a] = venue_wins.get(vk_a, 0) + (1 if win_a else 0)
        venue_total[venue] = venue_total.get(venue, 0) + 1

    return pd.DataFrame(rows)

rolling = compute_rolling_features(ms)
ms = pd.concat([ms.reset_index(drop=True), rolling], axis=1)
print("Rolling features added:", list(rolling.columns))
print(ms[['team_a', 'team_b', 'outcome', 'wr_a', 'wr_b', 'h2h_a']].head(5))


In [ ]:
# Step 6: Extract Powerplay & Death-Over strength from train_IPL.csv
# This is the "secret weapon" — teams with strong powerplay or explosive finishers
# tend to win by big margins, which is exactly what we need to predict.

train_ipl = loaded['train_ipl'].copy()

# Powerplay = first 6 overs (Over column 0 to 5)
pp = train_ipl[train_ipl['Over'] <= 5].groupby(['Match ID', 'Bat First']).agg(
    pp_runs=('Batter Runs', 'sum'),
    pp_wickets=('Wicket', 'sum')
).reset_index()

# Death overs = last 5 overs (Over 15-19)
death = train_ipl[train_ipl['Over'] >= 15].groupby(['Match ID', 'Bat First']).agg(
    death_runs=('Batter Runs', 'sum'),
    death_balls=('Valid Ball', 'sum')
).reset_index()
death['death_rr'] = death['death_runs'] / (death['death_balls'] / 6).clip(lower=0.1)

# Average per team across all their matches
team_pp = pp.groupby('Bat First').agg(
    avg_pp_runs=('pp_runs', 'mean'),
    avg_pp_wkts=('pp_wickets', 'mean')
).reset_index().rename(columns={'Bat First': 'team'})

team_death = death.groupby('Bat First').agg(
    avg_death_rr=('death_rr', 'mean')
).reset_index().rename(columns={'Bat First': 'team'})

team_power = team_pp.merge(team_death, on='team', how='outer')
print("Powerplay & Death-Over stats computed for", len(team_power), "teams")
print(team_power.sort_values('avg_pp_runs', ascending=False).head(5))


In [ ]:
# Step 7: Merge all features into one training-ready DataFrame

# Attach Team A power stats
ms = ms.merge(
    team_power.rename(columns={'team':'team_a','avg_pp_runs':'a_pp_runs',
                                'avg_pp_wkts':'a_pp_wkts','avg_death_rr':'a_death_rr'}),
    on='team_a', how='left'
)
# Attach Team B power stats
ms = ms.merge(
    team_power.rename(columns={'team':'team_b','avg_pp_runs':'b_pp_runs',
                                'avg_pp_wkts':'b_pp_wkts','avg_death_rr':'b_death_rr'}),
    on='team_b', how='left'
)

# Fill any missing power stats with the league average
for col in ['a_pp_runs','a_pp_wkts','a_death_rr','b_pp_runs','b_pp_wkts','b_death_rr']:
    ms[col] = ms[col].fillna(ms[col].mean())

# Encode teams and venues as numbers (ML models need numbers, not strings)
team_le = LabelEncoder()
venue_le = LabelEncoder()
team_le.fit(pd.concat([ms['team_a'], ms['team_b']]).unique())
venue_le.fit(ms['venue'].unique())

ms['ta_enc'] = team_le.transform(ms['team_a'])
ms['tb_enc'] = team_le.transform(ms['team_b'])
ms['v_enc']  = venue_le.transform(ms['venue'])
ms['yr']     = ms['date'].dt.year

# Temporal weighting — recent seasons matter more because teams evolve
ms['weight'] = ms['yr'].apply(
    lambda x: 4.0 if x >= 2025 else (3.0 if x >= 2023 else (2.0 if x >= 2020 else 1.0))
)

ms['venue_avg'] = ms.get('venue_avg_score', pd.Series([160]*len(ms))).fillna(160)

print("Final training DataFrame shape:", ms.shape)
print("Feature columns ready for model training.")


## 🤖 Model Training — XGBoost + Isotonic Calibration

We use **XGBoost** (gradient boosted trees) because:
- It handles non-linear interactions between features perfectly
- It natively supports multi-class predictions
- It's the go-to model for tabular data competitions

After training, we apply **Isotonic Calibration**:
> This fine-tunes the probability outputs so that when the model says "60% chance", 
> it actually happens 60% of the time. This is *critical* for Log-Loss scoring.

We evaluate using **5-Fold Cross Validation** — the honest, leakage-free way to measure performance.


In [ ]:
# Step 8: Train the championship model

FEATURES = [
    'ta_enc', 'tb_enc', 'v_enc', 'yr', 'venue_avg',
    'wr_a', 'wr_b', 'wr_diff',
    'h2h_a', 'h2h_edge',
    'v_wr_a',
    'a_pp_runs', 'a_pp_wkts', 'a_death_rr',
    'b_pp_runs', 'b_pp_wkts', 'b_death_rr'
]

X = ms[FEATURES].values
outcome_le = LabelEncoder()
y = outcome_le.fit_transform(ms['outcome'])

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# XGBoost with carefully tuned hyperparameters
base_xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

# Wrap with isotonic calibration (best for Log-Loss)
model = CalibratedClassifierCV(base_xgb, method='isotonic', cv=skf)

# Train on full data (with temporal weights)
print("Training model... (this takes ~2-3 minutes on CPU)")
model.fit(X_s, y, sample_weight=ms['weight'])
print("Training complete!")

# Honest OOF (Out-of-Fold) evaluation — the true indicator of leaderboard performance
print("\nRunning cross-validation for honest scoring...")
oof_probs = cross_val_predict(model, X_s, y, cv=skf, method='predict_proba')
oof_preds = oof_probs.argmax(axis=1)

print("=" * 45)
print(f"  CV Accuracy : {accuracy_score(y, oof_preds)*100:.2f}%")
print(f"  CV Log-Loss : {log_loss(y, oof_probs):.4f}")
print("=" * 45)
print("\nNote: Lower Log-Loss = better leaderboard ranking.")
print("Random guessing = 1.386 | Our target = below 1.30")


## 📊 Generate Leaderboard Submission

Now we apply the trained model to the **48 matches** in `public_lb_matches.csv`.
These are matches that already happened — but we don't have the outcomes.
The leaderboard judges our predicted probabilities against the real results.

**Review the table below before submitting.** Make sure probabilities for each row sum to 1.0.


In [ ]:
# Step 9: Predict on leaderboard matches and generate submission.csv

lb = loaded['lb_matches'].copy()

def predict_match(row):
    """Predict win probabilities for a single match using all available signals."""
    try:
        ta_enc = team_le.transform([row['team_a']])[0] if row['team_a'] in team_le.classes_ else 0
        tb_enc = team_le.transform([row['team_b']])[0] if row['team_b'] in team_le.classes_ else 0
        v_enc  = venue_le.transform([row['venue']])[0]  if row['venue']  in venue_le.classes_  else 0
    except:
        return [0.25, 0.25, 0.25, 0.25]

    # Use latest team stats for leaderboard prediction
    a_stats = team_power[team_power['team']==row['team_a']].iloc[0] if row['team_a'] in team_power['team'].values else pd.Series({'avg_pp_runs':50,'avg_pp_wkts':1.5,'avg_death_rr':9.5})
    b_stats = team_power[team_power['team']==row['team_b']].iloc[0] if row['team_b'] in team_power['team'].values else pd.Series({'avg_pp_runs':50,'avg_pp_wkts':1.5,'avg_death_rr':9.5})

    feat = np.array([[
        ta_enc, tb_enc, v_enc,
        pd.to_datetime(row['date']).year, 160,
        0.5, 0.5, 0.0,       # win rates (unknown for future)
        0.5, 0.0,             # h2h (unknown)
        0.5,                  # venue wr
        a_stats['avg_pp_runs'], a_stats['avg_pp_wkts'], a_stats['avg_death_rr'],
        b_stats['avg_pp_runs'], b_stats['avg_pp_wkts'], b_stats['avg_death_rr']
    ]])
    feat_s = scaler.transform(feat)
    return model.predict_proba(feat_s)[0]

print("Generating predictions for", len(lb), "leaderboard matches...")
results = []
for _, row in lb.iterrows():
    probs = predict_match(row)
    results.append({
        'match_id': row['match_id'],
        'A_small': round(float(probs[outcome_le.transform(['A_small'])[0]]), 6) if 'A_small' in outcome_le.classes_ else 0.25,
        'A_big':   round(float(probs[outcome_le.transform(['A_big'])[0]]),   6) if 'A_big'   in outcome_le.classes_ else 0.25,
        'B_small': round(float(probs[outcome_le.transform(['B_small'])[0]]), 6) if 'B_small' in outcome_le.classes_ else 0.25,
        'B_big':   round(float(probs[outcome_le.transform(['B_big'])[0]]),   6) if 'B_big'   in outcome_le.classes_ else 0.25,
    })

sub = pd.DataFrame(results)

print("\n👀 SUBMISSION PREVIEW (first 10 rows):")
display(sub.head(10))
print(f"\nProbability sum check (should be ~1.0): {(sub['A_small']+sub['A_big']+sub['B_small']+sub['B_big']).mean():.4f}")

sub.to_csv('submission.csv', index=False)
print("\nsubmission.csv saved and ready to submit!")
